# Loss functions and optimizers

_PyTorch_

**The two pieces that turn a model into learning: a loss to score it, an optimizer to fix it.**

Learning is a feedback loop. The loss answers "how wrong are we?" as a single number. Calling
        loss.backward() fills every parameter's .grad with the slope of that number
        with respect to that parameter. The optimizer then takes one downhill step.

       You attach the optimizer to the model's parameters once. From then on it remembers where they live and,
        for adaptive methods like Adam, keeps running statistics (a moving average of past gradients) to size each
        step automatically.

---

This notebook is a practice scaffold for the **Loss functions and optimizers** lesson. We unpack the feedback loop one step at a time — run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Build a tiny classification problem

We make 60 random points with 4 features each, and assign each one a class in `{0, 1, 2}`. The labels are stored as **integer class indices**, not one-hot rows — that is exactly what `CrossEntropyLoss` expects (it will index into the predicted scores using these integers).

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)  # reproducible

# --- tiny synthetic 3-class problem: 60 points, 4 features ---
N, D_in, C = 60, 4, 3
X = torch.randn(N, D_in)

# integer class labels 0/1/2 -- NOT one-hot. CrossEntropyLoss wants indices.
y = torch.randint(0, C, (N,))            # shape (N,), dtype long

### Step 2 — Define the model, loss, and optimizer

The model ends in a plain `nn.Linear` that emits **raw logits** of shape `(N, C)` — there is deliberately no softmax at the end, because `nn.CrossEntropyLoss` applies log-softmax internally. We hand the optimizer the model's parameters once; from then on `Adam` remembers where they live and keeps running statistics to size each step automatically.

In [ ]:
# --- model: outputs raw LOGITS of shape (N, C). NO softmax at the end! ---
model = nn.Sequential(
    nn.Linear(D_in, 16),
    nn.ReLU(),
    nn.Linear(16, C),                    # last layer -> raw logits, that's it
)

# loss + optimizer. Adam is given the model's parameters and a learning rate.
criterion = nn.CrossEntropyLoss()        # applies log-softmax INTERNALLY
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

### Step 3 — Train with the zero-grad / backward / step trio

Every PyTorch training step is the same three calls. `optimizer.zero_grad()` clears last step's gradients (PyTorch *adds* into `.grad`, so stale values would accumulate); `loss.backward()` fills every parameter's `.grad` via autograd; `optimizer.step()` takes one downhill step. We print the loss and training accuracy occasionally to watch it improve.

In [ ]:
for step in range(20):
    logits = model(X)                    # (N, C) raw logits
    loss = criterion(logits, y)          # pass logits + integer targets

    optimizer.zero_grad()                # 1. clear old grads (else they accumulate)
    loss.backward()                      # 2. backprop: fill every .grad
    optimizer.step()                     # 3. one downhill step

    if step % 4 == 0:
        acc = (logits.argmax(1) == y).float().mean().item()
        print(f"step {step:2d}  loss {loss.item():.4f}  train_acc {acc:.2f}")

### Step 4 — The CrossEntropyLoss gotcha, spelled out

Two mistakes break training silently. Applying `softmax` before `CrossEntropyLoss` softmaxes twice and flattens the signal; passing **one-hot** targets instead of integer indices is also wrong. The correct recipe is the one we used above: raw logits in, integer class indices as targets.

In [ ]:
# --- the GOTCHA, shown ---
# WRONG: softmax before CrossEntropyLoss -> double softmax, training degrades.
#   bad = nn.CrossEntropyLoss()(torch.softmax(logits, dim=1), y)
# WRONG: one-hot targets -> CrossEntropyLoss wants class indices, not one-hot.
#   y_onehot = torch.nn.functional.one_hot(y, C).float()  # do NOT pass this
# RIGHT: raw logits + integer indices, exactly as above.

print("done. note: the model has NO softmax layer -- CrossEntropyLoss adds it.")

## Visualize the data & results

_On the same tiny noisy problem, how do SGD and Adam compare step by step?_

### Step 1 — Build an ill-conditioned regression problem

To contrast SGD and Adam we need a problem where they behave differently. We scale the 8 feature columns by very different amounts (1 up to 20), so the loss surface is far steeper in some directions than others — exactly the situation where a single global learning rate struggles. The target is a noisy linear function of those features.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
N, d = 200, 8

# ill-conditioned features: columns scaled 1..20 so one direction is far steeper
feature_scales = np.array([1, 1, 3, 3, 8, 8, 20, 20.0])
X = rng.standard_normal((N, d)) * feature_scales

w_true = rng.standard_normal(d)
y = X @ w_true + 0.1 * rng.standard_normal(N)   # noisy linear target

### Step 2 — Define the loss and the two optimizers

`full_loss` is the mean-squared error over **all** the data — the curve we plot. `batch_grad` computes a noisy gradient on a random mini-batch, the way real training does. The `run` function drives either plain **SGD** (`w -= lr * g`) or **Adam**, which keeps moving averages of the gradient (`m`) and its square (`v`) to rescale each coordinate's step automatically.

In [ ]:
def full_loss(w):
    r = X @ w - y
    return 0.5 * np.mean(r * r)                  # MSE over ALL data (what we plot)

def batch_grad(w, idx):
    Xb, yb = X[idx], y[idx]
    return Xb.T @ (Xb @ w - yb) / len(idx)       # mini-batch gradient (noisy)

steps, bs, w0 = 50, 16, np.zeros(d)

def run(opt, lr):
    w = w0.copy()
    m = np.zeros(d)
    v = np.zeros(d)
    hist = []
    rs = np.random.default_rng(42)               # same batch stream for both
    for t in range(1, steps + 1):
        hist.append(full_loss(w))
        g = batch_grad(w, rs.integers(0, N, bs))
        if opt == "sgd":
            w = w - lr * g                        # plain SGD step
        else:                                     # Adam
            b1, b2, eps = 0.9, 0.999, 1e-8
            m = b1 * m + (1 - b1) * g
            v = b2 * v + (1 - b2) * g * g
            mh = m / (1 - b1**t)
            vh = v / (1 - b2**t)
            w = w - lr * mh / (np.sqrt(vh) + eps)
    return hist

### Step 3 — Run both and compare the final loss

We run each optimizer on the identical batch stream and compare where they end up. On this ill-conditioned problem Adam's per-coordinate step sizes let it reach a much lower loss than plain SGD in the same number of steps — which is why adaptive optimizers are the default for most deep nets.

In [ ]:
sgd = run("sgd",  0.0025)
adam = run("adam", 0.15)

print("final loss -- SGD:", round(sgd[-1], 3), " Adam:", round(adam[-1], 3))
# -> final loss -- SGD: 2.281  Adam: 1.008

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Build preds = torch.tensor([2.5, 0.0, 4.0]) and
            target = torch.tensor([3.0, 0.0, 5.0]). Compute nn.MSELoss()(preds, target).
            Predict the value by hand first (mean of the squared differences), then verify.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Instantiate the loss object, then call it: nn.MSELoss()(preds, target). — _MSELoss is a module; you build it once, then call it like a function._
- By hand: differences are [-0.5, 0, -1.0], squared [0.25, 0, 1.0], mean 0.4167. — _MSE is the mean of squared differences, so you can check the printed number._

**Answer:** import torch
import torch.nn as nn
preds  = torch.tensor([2.5, 0.0, 4.0])
target = torch.tensor([3.0, 0.0, 5.0])
print(nn.MSELoss()(preds, target))   # tensor(0.4167)
# (0.25 + 0.0 + 1.0) / 3 = 0.4167

</details>

**Problem 2.** Type this in Colab. The pitfall: nn.CrossEntropyLoss wants raw logits and
            integer class indices. With logits = torch.tensor([[2.0, 0.5, -1.0], [0.1, 1.5, 0.2]])
            and targets = torch.tensor([0, 1]), compute the loss. Then apply
            torch.softmax(logits, dim=1) first and pass that instead — show the loss is wrong (higher).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Call nn.CrossEntropyLoss()(logits, targets) with the raw logits. — _The loss applies log-softmax internally, so you must NOT softmax beforehand._
- Pass torch.softmax(logits, dim=1) instead and compare. — _Softmaxing twice (double softmax) flattens the values, giving a larger, wrong loss._

**Answer:** logits  = torch.tensor([[2.0, 0.5, -1.0], [0.1, 1.5, 0.2]])
targets = torch.tensor([0, 1])
ce = nn.CrossEntropyLoss()
print(ce(logits, targets))                       # tensor(0.3168)

</details>

**Problem 3.** Type this in Colab. Show that CrossEntropyLoss takes class indices, NOT one-hot rows.
            Reuse the logits above. Build the one-hot version of targets with
            torch.nn.functional.one_hot(targets, num_classes=3), try passing it, and observe the result;
            then pass the integer indices and print the loss.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Make one-hot targets with F.one_hot(targets, 3). — _This is what beginners wrongly hand to the loss; it is a (2, 3) tensor, not indices._
- Pass the integer index tensor targets instead. — _CrossEntropyLoss indexes the logits with integer class ids; that is the supported form._

**Answer:** import torch.nn.functional as F
oh = F.one_hot(targets, num_classes=3)           # tensor([[1,0,0],[0,1,0]])
# ce(logits, oh.float()) -> treats it as soft labels / errors in older versions; NOT what you want
print(ce(logits, targets))                       # tensor(0.3168)

</details>

**Problem 4.** Type this in Colab. The pitfall the optimizer hides: gradients accumulate. Make
            w = torch.tensor([1.0], requires_grad=True). Call (w * w).backward() twice in a row
            without zeroing, printing w.grad after each. Then zero with
            w.grad = None and do one clean backward.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Backward twice on w*w (derivative is 2w = 2). — _PyTorch ADDS each new gradient onto .grad, so it reads 2 then 4 — not 2 both times._
- Reset with w.grad = None (what optimizer.zero_grad() does) before the next backward. — _Clearing the grad makes the next step use only its own gradient._

**Answer:** w = torch.tensor([1.0], requires_grad=True)
(w * w).backward()
print(w.grad)        # tensor([2.])
(w * w).backward()
print(w.grad)        # tensor([4.])

</details>

**Problem 5.** Type this in Colab. Run the real three-step update once by hand. With torch.manual_seed(0),
            build model = nn.Linear(4, 1), opt = torch.optim.SGD(model.parameters(), lr=0.1),
            input x = torch.randn(8, 4), target y = torch.randn(8, 1). Do
            zero_grad &rarr; forward &rarr; MSELoss &rarr; backward &rarr;
            step, and print the loss before and after.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute and print the loss, then run opt.zero_grad(), loss.backward(), opt.step(). — _This is the canonical update trio every PyTorch model runs each step._
- Recompute the loss on the same batch after the step. — _One downhill SGD step should lower the loss on this batch._

**Answer:** torch.manual_seed(0)
model = nn.Linear(4, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
x = torch.randn(8, 4)
y = torch.randn(8, 1)
crit = nn.MSELoss()
loss = crit(model(x), y)
print(loss.item())                # 1.5046  (before)
opt.zero_grad()
loss.backward()
opt.step()
print(crit(model(x), y).item())   # 1.2473  (after -- lower)

</details>

**Problem 6.** Type this in Colab. Show why the optimizer must be given the model's parameters. Build
            model = nn.Linear(2, 1). Save a copy of its weight with
            w0 = model.weight.clone(). Create opt = torch.optim.SGD(model.parameters(), lr=0.5),
            run one update on x = torch.ones(3, 2), y = torch.zeros(3, 1), and check the weight
            actually changed.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Pass model.parameters() into the optimizer. — _Without it, step() updates nothing — the optimizer needs to know which tensors to move._
- Compare model.weight to w0 after the step. — _A nonzero difference confirms the optimizer is wired to the right parameters._

**Answer:** torch.manual_seed(0)
model = nn.Linear(2, 1)
w0 = model.weight.clone()
opt = torch.optim.SGD(model.parameters(), lr=0.5)
x, y = torch.ones(3, 2), torch.zeros(3, 1)
opt.zero_grad()
nn.MSELoss()(model(x), y).backward()
opt.step()
print(torch.allclose(model.weight, w0))   # False  -- weights moved

</details>

**Problem 7.** Type this in Colab. Train the tiny 3-class problem to convergence. With torch.manual_seed(0),
            make X = torch.randn(60, 4), y = torch.randint(0, 3, (60,)), and a model
            nn.Sequential(nn.Linear(4,16), nn.ReLU(), nn.Linear(16,3)) (raw logits, no softmax). Use
            CrossEntropyLoss + Adam(lr=1e-2) and run 30 steps, printing the loss every 10 steps.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Build the model with NO softmax layer at the end. — _CrossEntropyLoss adds log-softmax itself; a softmax layer would double it._
- Run the zero_grad/backward/step trio 30 times. — _Repeated downhill steps drive the loss down as the logits sharpen toward the labels._

**Answer:** torch.manual_seed(0)
X = torch.randn(60, 4)
y = torch.randint(0, 3, (60,))
model = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3))
crit = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for step in range(30):
    logits = model(X)
    loss = crit(logits, y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 10 == 0:
        print(step, round(loss.item(), 4))
# 0 1.1146
# 10 0.7714
# 20 0.5559

</details>

**Problem 8.** Type this in Colab. Attach a learning-rate scheduler. Build any
            opt = torch.optim.SGD(nn.Linear(2,2).parameters(), lr=0.1) and
            sched = torch.optim.lr_scheduler.StepLR(opt, step_size=1, gamma=0.5). Loop 4 times: call
            opt.step() then sched.step(), printing the current learning rate each round.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the rate with sched.get_last_lr() (or opt.param_groups[0]["lr"]). — _StepLR multiplies the rate by gamma each time you call sched.step()._
- Call sched.step() once per epoch, after opt.step(). — _Forgetting it leaves the rate frozen — the scheduler only moves when stepped._

**Answer:** opt = torch.optim.SGD(nn.Linear(2, 2).parameters(), lr=0.1)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=1, gamma=0.5)
for epoch in range(4):
    opt.step()
    sched.step()
    print(round(sched.get_last_lr()[0], 4))
# 0.05
# 0.025
# 0.0125
# 0.00625

</details>